![Your 6-week roadmap](../assets/roadmap/week2.png)

# Week 2 Exercise - Tune the Chunking and Beat the Baseline nDCG

This is the week 2 exercise. Take your time: it's an experiment, not a test, and there's no single right answer.

On Day 5 you measured GradeWise's retrieval: with BM25 over the pro chunks, the baseline is **nDCG@5 = 0.861** (MRR 0.840). You also saw where it struggles: `spanning` and `holistic` questions, whose answers don't fit in one chunk.

Asha's question for you: *"Is 1,000 characters actually the right chunk size for my notes?"* Your job is to find out, with numbers.

### 🎯 Your mission
1. 🔬 Run chunking experiments (`chunk_size`, `chunk_overlap`, `contextual_header`) and **beat nDCG@5 = 0.861**
2. 📊 Find out *which categories* improve and which get worse
3. ⚖️ Check the result is fair: bigger chunks mean the LLM reads more text, so compare at a similar context budget
4. 🧪 Bonus: write your own chunker and score it

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything here runs <b>offline and free</b>: each experiment builds a throw-away Chroma store in a temp folder with <code>DeterministicFakeEmbedding</code> and scores it with <b>BM25</b>, which ignores the embeddings, so <b>the scores are real</b>. They measure keyword retrieval only; Part 5 shows how to confirm your winner with real embeddings and hybrid search. Don't edit any files in <code>gradewise/</code> until Part 5.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

## Part 1: The experiment harness (provided)

Run this cell. It gives you two functions:
- `evaluate_chunking(chunk_size, chunk_overlap, contextual_header, k)` → chunk the four PDFs with those settings and score them on all 60 questions
- `evaluate_chunks(chunks, k)` → score *any* list of chunks you make yourself (for the bonus)

Each experiment takes about a second.

In [ ]:
import importlib.util
import sys
import tempfile
import uuid

import matplotlib.pyplot as plt
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR, REPO_ROOT
from gradewise.ingestion import chunk_documents, get_vectorstore, load_directory
from gradewise.rag import get_retriever

spec = importlib.util.spec_from_file_location("week2_eval", REPO_ROOT / "week2" / "evaluation" / "eval.py")
evaluation = importlib.util.module_from_spec(spec)
sys.modules["week2_eval"] = evaluation
spec.loader.exec_module(evaluation)

TESTS = evaluation.load_tests()
PAGES = load_directory(DATA_DIR / "knowledge-base")  # 💡 loaded once: only the chunking changes between runs
TMP = tempfile.mkdtemp(prefix="gradewise_w2_exercise_")


def evaluate_chunks(chunks: list[Document], k: int = 5) -> dict:
    """Score any list of chunks with BM25 on all 60 test questions.

    ⚠️ Offline: the Chroma store gets FAKE embeddings, but BM25 never uses them, so the scores are real.
    Returns eval.py's summary ({category: {"mrr", "ndcg", "coverage", "source", "n"}}) plus
    "context_chars": the average number of characters retrieved per question (what the LLM would read).
    """
    name = f"exp_{uuid.uuid4().hex[:8]}"
    store = get_vectorstore(name, embeddings=DeterministicFakeEmbedding(size=32), persist_dir=TMP)
    for i, chunk in enumerate(chunks):
        chunk.metadata.setdefault("chunk_index", i)
    store.add_documents(chunks, ids=[f"c{i}" for i in range(len(chunks))])
    retriever = get_retriever(store=store, k=k, mode="bm25")
    results, chars = [], 0
    for test in TESTS:
        docs = retriever.invoke(test.question)
        results.append(evaluation.evaluate_retrieval(test, docs, k))
        chars += sum(len(d.page_content) for d in docs)
    summary = evaluation.summarize(TESTS, results)
    summary["context_chars"] = chars / len(TESTS)
    return summary


def evaluate_chunking(
    chunk_size: int = 1000, chunk_overlap: int = 150, contextual_header: bool = True, k: int = 5
) -> dict:
    """One experiment: chunk the knowledge base with these settings, then score it."""
    chunks = chunk_documents(
        PAGES, chunk_size=chunk_size, chunk_overlap=chunk_overlap, contextual_header=contextual_header
    )
    summary = evaluate_chunks(chunks, k=k)
    summary["n_chunks"] = len(chunks)
    return summary


def row(name: str, summary: dict) -> dict:
    """Flatten a summary into one table row."""
    out = {"experiment": name, "chunks": summary.get("n_chunks"), "context_chars": round(summary["context_chars"])}
    out["MRR"] = round(summary["ALL"]["mrr"], 3)
    out["nDCG"] = round(summary["ALL"]["ndcg"], 3)
    for category in ("direct_fact", "numerical", "comparison", "spanning", "holistic"):
        out[f"nDCG·{category}"] = round(summary[category]["ndcg"], 2)
    return out


def show(rows: list[dict], sort_by: str | None = "nDCG") -> None:
    """Print rows as an aligned table, best first. (💡 a tiny stand-in for a pandas DataFrame)"""
    rows = sorted(rows, key=lambda r: r[sort_by], reverse=True) if sort_by else rows
    columns = list(rows[0])
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    print("  ".join("-" * widths[c] for c in columns))
    for r in rows:
        print("  ".join(str(r[c]).ljust(widths[c]) for c in columns))


print(f"✅ Harness ready: {len(TESTS)} questions, {len(PAGES)} pages, scratch stores in {TMP}")

## Part 2: Reproduce the baseline

First, make sure the harness agrees with `eval.py --offline --pipeline bm25` from Day 5. You should see nDCG = 0.861.

In [ ]:
baseline = evaluate_chunking()  # the defaults from gradewise/ingestion/chunking.py: 1000 / 150 / header on
results = [row("baseline 1000/150", baseline)]
show(results)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before running anything, predict: will <b>bigger</b> chunks (say 2,000 characters) raise or lower nDCG? What about <b>smaller</b> ones (500)? Write your guesses down. Then think about <i>why</i> keyword-based nDCG might be biased towards one of them.</span>
        </td>
    </tr>
</table>

## Part 3: Run your experiments

✋ **TODO:** add experiments to the list below and run the cell. Try at least six. Ideas:
- chunk sizes from 300 to 3,000
- overlap of 0, 10% and 20% of the chunk size
- `contextual_header=False` at the baseline size, to measure what the header is worth

In [ ]:
experiments = [
    # (chunk_size, chunk_overlap, contextual_header)
    (500, 100, True),
    # ✋ TODO: add at least five more experiments here
]

for size, overlap, header_on in experiments:
    summary = evaluate_chunking(size, overlap, header_on)
    results.append(row(f"{size}/{overlap}" + ("" if header_on else " no header"), summary))

show(results)

✋ **TODO:** plot nDCG against chunk size (one line, or one line per overlap). Here's a start:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter([r["chunks"] for r in results], [r["nDCG"] for r in results], color="#7c5cff")
for r in results:
    ax.annotate(r["experiment"], (r["chunks"], r["nDCG"]), fontsize=8)
ax.axhline(0.861, color="#ef4444", ls="--", label="baseline 0.861")
ax.set_xlabel("number of chunks")  # ✋ TODO: would chunk size or context_chars be a better x-axis?
ax.set_ylabel("nDCG@5")
ax.legend()
plt.show()

## Part 4: Which categories moved?

✋ **TODO:** compare your best experiment with the baseline **per category**. Which categories got better? Did any get worse? Write two or three sentences on why.

In [ ]:
# ✋ TODO: pick your best experiment and show baseline vs best for each category
# Hint: evaluation.print_summary({"baseline": baseline, "best": best_summary}, k=5)
#       prints Day 5's "before → after" table. Pass only the per-category entries:
#       {c: v for c, v in summary.items() if isinstance(v, dict)}

✍️ *Your analysis here:*

## Part 5: Is it a fair win?

Look at the `context_chars` column: the average number of characters retrieved per question, which is what the LLM has to read (and you pay for). Bigger chunks contain more keywords *just by being bigger*, so a higher nDCG at k = 5 might simply mean "we sent more text".

✋ **TODO:** compare two set-ups with a **similar context budget**, for example 500-character chunks with `k=10` versus 1,000-character chunks with `k=5`. Use `evaluate_chunking(..., k=10)`. Which wins now?

In [ ]:
# ✋ TODO: equal-budget comparison

### Confirm it with real embeddings (needs a key)

BM25 is only half of GradeWise's hybrid retriever. To check that your winner helps the real pipeline too:
1. Edit `CHUNK_SIZE` / `CHUNK_OVERLAP` in `gradewise/ingestion/chunking.py`
2. Re-ingest from scratch: `uv run week2/pro_implementation/ingest.py --reset`
3. Evaluate the pro (hybrid) pipeline: `uv run week2/evaluation/eval.py`
4. Put the defaults back afterwards (or keep them, if the numbers say so!)

## Part 6: Bonus, your own chunker

✋ **TODO:** write a chunker of your own and score it with `evaluate_chunks(my_chunks)`. Ideas: one chunk per `##` section (however long), a sliding window of 5 sentences, or the default chunks *plus* one extra "summary" chunk per section.

In [ ]:
def my_chunker(pages: list[Document]) -> list[Document]:
    # ✋ TODO: return a list of Documents, each with at least "source" and "page" in its metadata
    return chunk_documents(pages)


mine = evaluate_chunks(my_chunker(PAGES))
print(f"My chunker: nDCG = {mine['ALL']['ndcg']:.3f}, context = {mine['context_chars']:.0f} chars/question")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">This loop (change one thing, re-measure, look per category, check the cost) is the day-to-day work of an AI engineer. Being able to say "1,500-character chunks raise nDCG from 0.86 to 0.88, mostly on big-picture questions, for 17% more prompt text" is what gets a change approved. "It felt better" doesn't.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The settings you're tuning are <code>CHUNK_SIZE</code> and <code>CHUNK_OVERLAP</code> at the top of <code>gradewise/ingestion/chunking.py</code>, and the metrics are <code>calculate_mrr</code> / <code>calculate_ndcg</code> in <code>week2/evaluation/eval.py</code>. Whatever you choose here is what GradeWise's grader will search when it looks for evidence in weeks 4 and 5.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Submit and compare</h2>
            <span style="color:#f59e0b;">When you're done, compare with <code>week2/solutions/week2_EXERCISE_SOLUTION.ipynb</code>. Proud of your chunker? Share it in <code>community_contributions/</code> (clear the outputs first: see <code>CONTRIBUTING.md</code>). Then on to <b>week 3</b>, where GradeWise becomes an agent.</span>
        </td>
    </tr>
</table>